In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import time, os, sys
from IPython.display import clear_output

import keysight.qcs as qcs
from keysight.qcs.channels import  ConstantEnvelope,  RFWaveform
from keysight.qcs.variables import Array, Scalar

from preamble.pump_utils import *

us,ns,MHz,GHz=1e-6,1e-9,1e6,1e9

In [2]:
# port='socket://163.152.38.107:4000'
port='COM3'

pump = PumpController(port=port)

In [3]:
LO_freq=6.1*GHz

readout_awg_address = qcs.Address(chassis = 1, slot = 8, channel = 1)
dnc_address = qcs.Address(1, 17, 2)
dig_address = qcs.Address(1, 18, 2)

readout_awg = qcs.Channels(range(1), "readout_channels", absolute_phase = True)
digs = qcs.Channels(range(1), "acquire_channels", absolute_phase = True)

In [ ]:
mapper = qcs.ChannelMapper('localhost')
mapper.add_channel_mapping(readout_awg, [readout_awg_address], qcs.InstrumentEnum.M5300AWG)
mapper.add_channel_mapping(digs, [dig_address], qcs.InstrumentEnum.M5200Digitizer)
mapper.add_downconverters([dig_address], [dnc_address])
mapper.set_lo_frequencies(readout_awg_address, LO_freq)
mapper.set_lo_frequencies(dnc_address, LO_freq)

backend = qcs.HclBackend(mapper, fpga_postprocessing=True,init_time=2*us)
backend._keep_progress_bar=False
backend.is_system_ready()

In [5]:
def readout_program(freq_array,duration=1*us,amplitude=0.02,shots=1000):

    readout_freq = qcs.Scalar("readout_freq", dtype=float, value=LO_freq)
    readout_pulse = RFWaveform(duration, ConstantEnvelope(), amplitude, readout_freq)
    integration_filter = RFWaveform(duration, ConstantEnvelope(), 1, readout_freq)

    program = qcs.Program()
    program.n_shots(shots)
    program.add_waveform(readout_pulse,readout_awg)
    program.add_acquisition(integration_filter,digs)

    sweep=qcs.Array('freq_array',value=freq_array)
    program.sweep(sweep,readout_freq)
    
    return program

# Abort program

In [ ]:
current_id=backend.get_program_execution_history()[0]['accession_id']
backend.abort_program(current_id)

# Gain measurements

In [75]:
# signal_freqs=np.array([6.543,6.594,6.651,6.684,6.713,6.746,6.780,6.831])*GHz

signal_freqs=np.array([6.543,6.594,6.651])*GHz


pump_powers = np.linspace(2, 10, 11)
pump_freqs = np.linspace(7.4, 7.6, 11)*GHz

duration= 4*us
amplitude=0.02
n_shots=2000


In [ ]:
# Reference signal measurement
program=readout_program(signal_freqs,duration=duration,amplitude=amplitude,shots=n_shots*4)
pump.connect().off(ch=1)
result=qcs.Executor(backend).execute(program)
reference=np.abs(result.get_iq_array(avg=True).to_numpy())[0]
pump.close_all()

In [77]:
# Amplified signal measurement
program=readout_program(signal_freqs,duration=duration,amplitude=amplitude,shots=n_shots)

wait = 0.5 # Seconds
pump.connect().on(ch=1)
amplifed = np.zeros((len(pump_powers), len(pump_freqs), len(signal_freqs)), dtype=float)

for i, pump_power in enumerate(pump_powers):
    for j, pump_freq in enumerate(pump_freqs):
        print(f'progress: {i*len(pump_freqs)+j+1}/{len(pump_powers)*len(pump_freqs)}')

        pump.set(1,pump_freq,pump_power)
        time.sleep(wait)

        result=qcs.Executor(backend).execute(program)
        amplifed[i,j,:]=np.abs(result.get_iq_array(avg=True).to_numpy())[0]
        clear_output()

pump.off(ch=1).close_all()

In [78]:
# Gain estimation

gain = np.zeros((len(pump_powers), len(pump_freqs), len(signal_freqs)), dtype=float)
gain_mean = np.zeros((len(pump_powers), len(pump_freqs)), dtype=float)
gain_min = np.zeros((len(pump_powers), len(pump_freqs)), dtype=float)

for i, pump_power in enumerate(pump_powers):
    for j, pump_freq in enumerate(pump_freqs):
        gain[i,j,:]=amplifed[i,j,:]/reference
        gain_mean[i,j]=np.mean(gain[i,j])
        gain_min[i,j]=np.min(gain[i,j])

# Plot & Data save

In [79]:
mode = "min" # "mean" or "min"
tag=  "TWPA_optimize"
save_dir = "data"

if mode =="mean":
    gain_matrix=gain_mean
elif mode =="min":
    gain_matrix=gain_min

if not os.path.exists(save_dir):
    os.makedirs(save_dir)

timestamp = time.strftime("%y%m%d_%H%M%S")
out_png_path = os.path.join(save_dir, f"{tag}_{mode}_{timestamp}.png")
out_data_path = os.path.join(save_dir, f"{tag}_{timestamp}.nc")

In [ ]:
opt_idx = np.unravel_index(np.argmax(gain_matrix), gain_matrix.shape)
opt_power_dbm = float(pump_powers[opt_idx[0]])
opt_freq_hz = float(pump_freqs[opt_idx[1]])
gain_opt=np.round(20 * np.log10(gain[np.unravel_index(np.argmax(gain_matrix), gain_matrix.shape)]),2)

plt.figure(figsize=(8, 4))
extent = [pump_freqs[0] / GHz, pump_freqs[-1] / GHz,
            pump_powers[0], pump_powers[-1]]

eps = np.finfo(float).tiny
Z = 20 * np.log10(gain_matrix + eps)
im = plt.imshow(Z, origin="lower", extent=extent, aspect="auto", cmap="inferno")

plt.colorbar(im, label="Gain (dB)")

plt.scatter(opt_freq_hz / 1e9, opt_power_dbm, marker="x", color="red")
mode_label = "MEAN" if mode == "mean" else "MIN (worst-case)"
freqs_str = ", ".join(f"{f/1e9:.3f}" for f in signal_freqs)
plt.title( f"Signal freqs (GHz): {freqs_str}\n"
           f"[{mode_label}] Opt: {opt_freq_hz/1e9:.5f} GHz, {opt_power_dbm:.2f} dBm | "
            f"Gain: {gain_opt} dB")
plt.xlabel("Pump Frequency (GHz)")
plt.ylabel("Pump Power (dBm)")
plt.tight_layout()
plt.savefig(out_png_path, dpi=200)

In [ ]:
# Save data
da = xr.DataArray(gain, 
                  coords=[pump_freqs, pump_powers, signal_freqs], dims=["pump_freqs", "pump_powers", "signal_freqs"])
da.to_netcdf(out_data_path)

# Data load

In [149]:
loaded_da = xr.open_dataarray("data/TWPA_optimize_260623_223522.nc")
loaded_gain_matrix=np.min(loaded_da[:,:],axis=2).to_numpy()
# loaded_gain_matrix=np.mean(loaded_da[:,:],axis=2).to_numpy()

loaded_pump_freqs=loaded_da.pump_freqs
loaded_pump_powers=loaded_da.pump_powers
loaded_signal_freqs=loaded_da.signal_freqs

In [ ]:
opt_idx = np.unravel_index(np.argmax(loaded_gain_matrix), loaded_gain_matrix.shape)
opt_power_dbm = float(loaded_pump_powers[opt_idx[0]])
opt_freq_hz = float(loaded_pump_freqs[opt_idx[1]])
gain_opt=np.round(20 * np.log10(loaded_da[np.unravel_index(np.argmax(loaded_gain_matrix), loaded_gain_matrix.shape)]),2).to_numpy()

plt.figure(figsize=(8, 4))
extent = [loaded_pump_freqs[0] / 1e9, loaded_pump_freqs[-1] / 1e9,
            loaded_pump_powers[0], loaded_pump_powers[-1]]

eps = np.finfo(float).tiny
Z = 20 * np.log10(loaded_gain_matrix + eps)
im = plt.imshow(Z, origin="lower", extent=extent, aspect="auto", cmap="inferno")

plt.colorbar(im, label="Gain (dB)")

plt.scatter(opt_freq_hz / 1e9, opt_power_dbm, marker="x", color="red")
mode_label = "MEAN" if mode == "mean" else "MIN (worst-case)"
freqs_str = ", ".join(f"{f/1e9:.3f}" for f in signal_freqs)
plt.title( f"Signal freqs (GHz): {freqs_str}\n"
           f"[{mode_label}] Opt: {opt_freq_hz/1e9:.5f} GHz, {opt_power_dbm:.2f} dBm | "
            f"Gain: {gain_opt} dB")
plt.xlabel("Pump Frequency (GHz)")
plt.ylabel("Pump Power (dBm)")
plt.tight_layout()

plt.show()
